# DP-HPO: Full 25-Seed Experiment Runner

**Before running:** Runtime → Change runtime type → **T4 GPU**

Estimated time on T4 GPU: **8–14 hours** (vs ~150 hrs CPU).
Leave it running overnight. Results saved to `results_v2.json` and auto-downloaded.

---

## Cell 1 — Install dependencies

In [ ]:
!pip install -q optuna scikit-optimize scipy scikit-learn numpy
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
print('Installation complete.')

## Cell 2 — Pull dp_hpo.py from GitHub

In [ ]:
!wget -q https://raw.githubusercontent.com/kartikjsonawane/dp-hpo/master/dp_hpo.py
!python -c "import dp_hpo; print('dp_hpo.py loaded OK — methods:', list(dir(dp_hpo)))[:5]"
print('dp_hpo.py ready.')

## Cell 3 — Configuration

In [ ]:
# ── CONFIG — change these if needed ──────────────────────────────────────────
N_SEEDS   = 25          # Full run. Set to 5 for a quick sanity check.
SUBSAMPLE = 10000       # Samples per seed for large datasets (MNIST etc.)
# ─────────────────────────────────────────────────────────────────────────────

SEEDS = list(range(N_SEEDS))
N_BASELINES = 8
print(f'Seeds: {N_SEEDS}  |  Subsample: {SUBSAMPLE}  |  Baselines: {N_BASELINES}')

## Cell 4 — Imports & helpers

In [ ]:
import numpy as np
import json, time, itertools, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer, fetch_openml
from scipy.stats import wilcoxon as scipy_wilcoxon

from dp_hpo import (run_all_methods, dp_hpo, dp_hpo_ordered,
                    hyperband_search, bohb_search, smac_search,
                    random_search_k10, DIMS, VALS, DEFAULTS)

# GPU strategy
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'Running on: {len(gpus)} GPU(s)' if gpus else 'Running on CPU')


def prep(X, y, seed, subsample=None):
    if subsample and len(X) > subsample:
        idx = np.random.RandomState(seed).choice(len(X), subsample, replace=False)
        X, y = X[idx], y[idx]
    Xr, Xv, yr, yv = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    sc = StandardScaler()
    Xr = sc.fit_transform(Xr).astype(np.float32)
    Xv = sc.transform(Xv).astype(np.float32)
    return Xr, yr, Xv, yv


def wilcoxon_test(accs_a, accs_b, name_b='baseline', bonferroni_m=32):
    diffs = np.array(accs_a) - np.array(accs_b)
    if np.all(diffs == 0):
        return {'W': None, 'p': None, 'significant': False, 'effect_size_d': 0.0}
    stat, p = scipy_wilcoxon(accs_a, accs_b, alternative='two-sided')
    d_eff   = float(np.mean(diffs) / (np.std(diffs) + 1e-12))
    alpha_adj = 0.05 / bonferroni_m
    sig = p < alpha_adj
    print(f'  vs {name_b:<28} p={p:.4f}  d={d_eff:+.3f}  {"SIG" if sig else "ns"}')
    return {'W': float(stat), 'p': float(p), 'significant': bool(sig), 'effect_size_d': d_eff}

print('Helpers loaded.')

## Cell 5 — Load all datasets (once)

In [ ]:
print('Loading datasets...')

# 1. UCI Breast Cancer
X_bc, y_bc = load_breast_cancer(return_X_y=True)
print(f'  Breast Cancer: {X_bc.shape}')

# 2. MNIST
(X_mtr, y_mtr), (X_mte, y_mte) = tf.keras.datasets.mnist.load_data()
X_mnist = np.concatenate([X_mtr, X_mte]).reshape(-1, 784).astype(float) / 255.0
y_mnist = np.concatenate([y_mtr, y_mte])
print(f'  MNIST: {X_mnist.shape}')

# 3. Fashion-MNIST
(X_ftr, y_ftr), (X_fte, y_fte) = tf.keras.datasets.fashion_mnist.load_data()
X_fashion = np.concatenate([X_ftr, X_fte]).reshape(-1, 784).astype(float) / 255.0
y_fashion = np.concatenate([y_ftr, y_fte])
print(f'  Fashion-MNIST: {X_fashion.shape}')

# 4. Adult Income
import pandas as pd
data = fetch_openml('adult', version=2, as_frame=True, parser='auto')
df   = data.frame
X_df = pd.get_dummies(df.drop(columns=['class']), drop_first=True)
X_adult = X_df.values.astype(float)
y_adult = (df['class'].astype(str).str.strip() == '>50K').astype(int).values
print(f'  Adult Income: {X_adult.shape}')

DATASETS = {
    'UCI Breast Cancer': (X_bc,     y_bc,     None),
    'MNIST':             (X_mnist,  y_mnist,  SUBSAMPLE),
    'Fashion-MNIST':     (X_fashion,y_fashion,SUBSAMPLE),
    'Adult Income':      (X_adult,  y_adult,  SUBSAMPLE),
}
print('All datasets ready.')

## Cell 6 — Main experiment loop

This is the long cell. It saves a checkpoint `results_v2_checkpoint.json` after each dataset so you don't lose progress if the session times out.

In [ ]:
METHODS_LIST = [
    'Grid Search', 'Random Search (k=20)', 'Random Search (k=10)',
    'Bayesian Opt', 'Optuna (TPE)', 'Hyperband', 'BOHB', 'SMAC', 'DP-HPO (Proposed)'
]
BASELINES = [m for m in METHODS_LIST if m != 'DP-HPO (Proposed)']
bonferroni_m = N_BASELINES * len(DATASETS)
alpha_adj    = 0.05 / bonferroni_m

all_results     = {}
wall_clock_start = time.time()

for ds_name, (X, y, subsample) in DATASETS.items():
    print(f'\n{"="*65}')
    print(f'Dataset: {ds_name}  |  N_SEEDS={N_SEEDS}  |  subsample={subsample}')
    print('='*65)

    ds_results = {m: {'accs': [], 'evals': [], 'times': []} for m in METHODS_LIST}
    ds_start   = time.time()

    for seed in SEEDS:
        seed_start = time.time()
        print(f'\n  Seed {seed}/{N_SEEDS-1}:')
        Xr, yr, Xv, yv = prep(X, y, seed, subsample)

        run = run_all_methods(Xr, yr, Xv, yv, random_state=seed)
        for method, vals in run.items():
            if method in ds_results:
                ds_results[method]['accs'].append(vals['accuracy'])
                ds_results[method]['evals'].append(vals['n_evaluations'])
                ds_results[method]['times'].append(vals['time_seconds'])

        elapsed = time.time() - seed_start
        print(f'  Seed {seed} done in {elapsed:.0f}s')

    # ── Summary table ────────────────────────────────────────────────────────
    print(f'\n  {"Method":<32} {"Mean Acc%":>10}  {"Std%":>6}  {"Evals":>6}  {"Time(s)":>8}')
    print('  ' + '-'*70)
    for m in METHODS_LIST:
        v = ds_results[m]
        if not v['accs']: continue
        ma = np.mean(v['accs']); sa = np.std(v['accs'])
        me = np.mean(v['evals']); mt = np.mean(v['times'])
        print(f'  {m:<32} {ma*100:>9.2f}%  {sa*100:>5.2f}  {me:>6.0f}  {mt:>8.1f}s')

    # ── Wilcoxon tests ───────────────────────────────────────────────────────
    print(f'\n  Wilcoxon vs DP-HPO (Bonferroni α_adj={alpha_adj:.5f}):')
    dp_accs = ds_results['DP-HPO (Proposed)']['accs']
    sig_results = {}
    for b in BASELINES:
        if ds_results[b]['accs']:
            sig_results[b] = wilcoxon_test(dp_accs, ds_results[b]['accs'],
                                           name_b=b, bonferroni_m=bonferroni_m)

    # ── Store ────────────────────────────────────────────────────────────────
    all_results[ds_name] = {
        m: {
            'accs':       [float(x) for x in v['accs']],
            'evals':      [int(x)   for x in v['evals']],
            'times':      [float(x) for x in v['times']],
            'mean_acc':   float(np.mean(v['accs']))  if v['accs'] else None,
            'std_acc':    float(np.std(v['accs']))   if v['accs'] else None,
            'mean_evals': float(np.mean(v['evals'])) if v['evals'] else None,
            'mean_time':  float(np.mean(v['times'])) if v['times'] else None,
        }
        for m, v in ds_results.items()
    }
    all_results[ds_name]['_wilcoxon'] = sig_results

    ds_elapsed = time.time() - ds_start
    print(f'\n  Dataset done in {ds_elapsed/3600:.2f} hours')

    # ── Checkpoint save after each dataset ──────────────────────────────────
    checkpoint = {
        'version': '2.0', 'n_seeds': N_SEEDS, 'seeds': SEEDS,
        'bonferroni_m': bonferroni_m, 'alpha_adj': alpha_adj,
        'main_results': all_results,
        'datasets_completed': list(all_results.keys()),
    }
    with open('results_v2_checkpoint.json', 'w') as f:
        json.dump(checkpoint, f, indent=2)
    print(f'  Checkpoint saved: results_v2_checkpoint.json')

total_hours = (time.time() - wall_clock_start) / 3600
print(f'\n{"="*65}')
print(f'ALL DATASETS COMPLETE  |  Total wall-clock: {total_hours:.2f} hours')
print('='*65)

## Cell 7 — Save final results_v2.json

In [ ]:
output = {
    'version':      '2.0',
    'n_seeds':      N_SEEDS,
    'seeds':        SEEDS,
    'bonferroni_m': bonferroni_m,
    'alpha_adj':    alpha_adj,
    'main_results': all_results,
}

with open('results_v2.json', 'w') as f:
    json.dump(output, f, indent=2)

print('Saved: results_v2.json')
print(f'Size: {os.path.getsize("results_v2.json") / 1024:.1f} KB')

# Print final summary
print('\n── Final Results Summary ──────────────────────────────────────')
for ds_name, ds_data in all_results.items():
    print(f'\n{ds_name}')
    for m in METHODS_LIST:
        if m in ds_data and ds_data[m].get('mean_acc') is not None:
            mean_pct = ds_data[m]['mean_acc'] * 100
            std_pct  = ds_data[m]['std_acc']  * 100
            evals    = ds_data[m]['mean_evals']
            print(f'  {m:<32} {mean_pct:.2f}% ± {std_pct:.2f}%  ({evals:.0f} evals)')

## Cell 8 — Download results_v2.json to your computer

In [ ]:
import os
from google.colab import files
files.download('results_v2.json')
print('Download triggered. Check your browser downloads folder.')

## Cell 9 — (Optional) Ablation A1: All 24 orderings

Run this separately after the main loop. Uses Breast Cancer only, 5 seeds.

In [ ]:
print('Ablation A1: All 4! = 24 dimension orderings')
print('Dataset: UCI Breast Cancer, Seeds 0-4\n')

ordering_results = {}
for seed in range(5):
    Xr_a, yr_a, Xv_a, yv_a = prep(X_bc, y_bc, seed)
    seed_res = {}
    for perm in itertools.permutations(range(len(DIMS))):
        label = '→'.join(DIMS[i] for i in perm)
        acc, n_ev = dp_hpo_ordered(Xr_a, yr_a, Xv_a, yv_a,
                                   order=list(perm), random_state=seed)
        seed_res[label] = {'order': list(perm), 'accuracy': float(acc), 'n_evaluations': int(n_ev)}
    ordering_results[f'seed_{seed}'] = seed_res
    print(f'  Seed {seed}: best={max(v["accuracy"] for v in seed_res.values()):.4f}  '
          f'worst={min(v["accuracy"] for v in seed_res.values()):.4f}')

with open('ablation_a1_orderings.json', 'w') as f:
    json.dump(ordering_results, f, indent=2)

from google.colab import files
files.download('ablation_a1_orderings.json')
print('\nAblation A1 complete. Downloaded ablation_a1_orderings.json')